# Análisis de Estabilidad de Talud — Criterio de Mohr-Coulomb & Método de Bishop Simplificado

---

**Universidad Nacional de Colombia sede Manizales**

---

## El problema

Se tiene un **talud homogéneo** de altura $H = 10$ m con inclinación $2H:1V$. El suelo presenta las siguientes propiedades:

| Parámetro | Símbolo | Valor | Unidad |
|-----------|---------|-------|--------|
| Peso unitario | $\\gamma$ | 18 | kN/m³ |
| Cohesión efectiva | $c'$ | 20 | kPa |
| Ángulo de fricción interno | $\\phi'$ | 25 | ° |

El nivel freático se encuentra **por debajo de la base** del talud, por lo que **no hay presiones de poro** ($u = 0$).

### Geometría del dominio

```
y(m)
10 |████████████|                    |
   |  PLATAFORMA|  TALUD (2H:1V)    | PLATAFORMA
   |   ALTA     |\                  |   BAJA
 0 |____________| \________________ |___________
   0           20  \  40           60
                    x(m)
```

- Plataforma alta: $x \\in [0, 20]$ m, $y = 10$ m
- Talud: $x \\in [20, 40]$ m (descenso de 10 m en 20 m → pendiente $1V:2H$)
- Plataforma baja: $x \\in [40, 60]$ m, $y = 0$ m

### Método de solución: Bishop Simplificado

El **Factor de Seguridad (FS)** se obtiene de forma iterativa:

$$FS = \\frac{\\displaystyle\\sum_{i}\\frac{c'\\,\\Delta x_i + W_i\\tan\\phi'}{m_{\\alpha,i}}}{\\displaystyle\\sum_{i} W_i \\sin\\alpha_i}$$

donde:

$$m_{\\alpha,i} = \\cos\\alpha_i + \\frac{\\sin\\alpha_i\\,\\tan\\phi'}{FS}$$

y el ángulo $\\alpha_i$ sigue la convención:

$$\\sin\\alpha_i = \\frac{x_c - x_{m,i}}{R}$$

> **Convención de signos:** $\\sin\\alpha_i > 0$ para dovelas a la izquierda del centro (zona activa, contribuyen al deslizamiento); $\\sin\\alpha_i < 0$ para dovelas a la derecha (zona pasiva, se oponen).

### Criterio de Mohr-Coulomb

$$\\tau_f = c' + \\sigma'_n \\tan\\phi'$$

donde el esfuerzo normal efectivo en la base de la dovela $i$ es:

$$\\sigma'_{n,i} = \\frac{W_i}{\\Delta x_i} \\cos^2\\alpha_i$$

## Importamos librerías

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.cm as cm
import warnings
warnings.filterwarnings('ignore')

# Estilo general de las gráficas
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3
})

print('Librerías importadas correctamente.')

## Parámetros del problema

In [ ]:
# ── Propiedades del suelo ──────────────────────────────────────────────────
gamma   = 18.0          # Peso unitario [kN/m³]
c_prima = 20.0          # Cohesión efectiva [kPa]
phi_deg = 25.0          # Ángulo de fricción efectivo [°]
phi     = np.radians(phi_deg)

# ── Geometría del talud ───────────────────────────────────────────────────
H      = 10.0   # Altura del talud [m]
L_plat = 20.0   # Longitud de cada plataforma [m]
L_tal  = 20.0   # Longitud horizontal del talud [m] (2H:1V → 20 m horiz / 10 m vert)

# Coordenadas clave del perfil
x_ini_talud = L_plat                    # 20 m
x_fin_talud = L_plat + L_tal           # 40 m
x_max       = L_plat + L_tal + L_plat  # 60 m

# ── Discretización de dovelas ─────────────────────────────────────────────
n_dovelas = 20   # Número de dovelas

print('=' * 52)
print('  PARÁMETROS DEL PROBLEMA')
print('=' * 52)
print(f'  Altura del talud:        H  = {H:.1f} m')
print(f'  Inclinación:             2H : 1V')
print(f'  Peso unitario:           γ  = {gamma:.1f} kN/m³')
print(f'  Cohesión efectiva:       c\' = {c_prima:.1f} kPa')
print(f'  Ángulo de fricción:      φ\' = {phi_deg:.1f}°')
print(f'  Presión de poro:         u  = 0 (NF bajo la base)')
print(f'  Número de dovelas:       n  = {n_dovelas}')
print('=' * 52)

## Funciones auxiliares

In [ ]:
def perfil_terreno(x):
    """
    Devuelve la elevación y del perfil del terreno.
    Acepta escalar o array numpy.
    """
    if np.isscalar(x):
        if x <= x_ini_talud:
            return H
        elif x <= x_fin_talud:
            return H - (x - x_ini_talud) * (H / L_tal)
        else:
            return 0.0
    else:
        return np.where(
            x <= x_ini_talud, H,
            np.where(x <= x_fin_talud,
                     H - (x - x_ini_talud) * (H / L_tal),
                     0.0)
        )


def interseccion_circulo_base(xc, yc, R):
    """Límites x del círculo sobre y = 0."""
    if R < abs(yc):
        return None, None
    dx = np.sqrt(R**2 - yc**2)
    return xc - dx, xc + dx


def factor_seguridad_bishop(xc, yc, R, n=20, tol=1e-6, max_iter=200):
    """
    Factor de Seguridad por Bishop Simplificado con criterio Mohr-Coulomb.

    Convención de signos (fundamental):
        sin(α_i) = (xc - xm_i) / R
        → positivo para dovelas a la IZQUIERDA del centro (zona activa).
        El denominador Σ W·sin(α) debe ser > 0.

    Retorna
    -------
    FS      : float o None
    dovelas : lista de dicts con resultados por dovela
    hist_FS : lista con historial de convergencia
    """
    xl, xr = interseccion_circulo_base(xc, yc, R)
    if xl is None:
        return None, None, None

    xl = max(xl, 0.0)
    xr = min(xr, x_max)
    if xr - xl < 1.0:
        return None, None, None

    dx = (xr - xl) / n
    xm = np.array([xl + (i + 0.5) * dx for i in range(n)])

    # Elevación del terreno
    y_ter = np.array([perfil_terreno(x) for x in xm])

    # Elevación del arco circular (parte inferior)
    disc = R**2 - (xm - xc)**2
    if np.any(disc < 0):
        return None, None, None
    y_fal = yc - np.sqrt(disc)

    # Altura de cada dovela
    h_i = y_ter - y_fal
    valid = h_i > 0
    if valid.sum() < 3:
        return None, None, None

    xm_v   = xm[valid]
    h_i_v  = h_i[valid]
    y_fal_v= y_fal[valid]
    y_ter_v= y_ter[valid]

    # Peso de cada dovela [kN/m]
    W_i = gamma * h_i_v * dx

    # Ángulo α — CONVENCIÓN CORRECTA: sin(α) = (xc - xm) / R
    # Positivo para dovelas a la izquierda del centro (generan momento motor)
    sin_a = np.clip((xc - xm_v) / R, -1.0, 1.0)
    cos_a = np.sqrt(1.0 - sin_a**2)

    # Denominador del FS: debe ser positivo
    den = np.sum(W_i * sin_a)
    if den <= 0:
        return None, None, None

    # Iteración de Bishop
    FS = 1.5
    hist_FS = [FS]

    for _ in range(max_iter):
        m_alpha = cos_a + (sin_a * np.tan(phi)) / FS
        if np.any(m_alpha <= 0):
            return None, None, None

        num   = np.sum((c_prima * dx + W_i * np.tan(phi)) / m_alpha)
        FS_nw = num / den
        hist_FS.append(FS_nw)

        if abs(FS_nw - FS) < tol:
            FS = FS_nw
            break
        FS = FS_nw
    else:
        return None, None, None  # no convergió

    if FS <= 0 or FS > 20:
        return None, None, None

    # Esfuerzos en la base de cada dovela
    sigma_n = (W_i / dx) * cos_a**2
    tau_f   = c_prima + sigma_n * np.tan(phi)

    dovelas = [
        {
            'x'    : xm_v[i],
            'h'    : h_i_v[i],
            'W'    : W_i[i],
            'sin_a': sin_a[i],
            'cos_a': cos_a[i],
            'alpha': float(np.degrees(np.arcsin(np.clip(sin_a[i], -1, 1)))),
            'sigma': sigma_n[i],
            'tau'  : tau_f[i],
            'y_fal': y_fal_v[i],
            'y_ter': y_ter_v[i],
            'dx'   : dx
        }
        for i in range(len(xm_v))
    ]

    return FS, dovelas, hist_FS


print('✓ Funciones auxiliares definidas correctamente.')

## Búsqueda del círculo crítico

Se recorre una grilla de centros $(x_c, y_c)$ y radios $R$ para encontrar la combinación que produce el **mínimo Factor de Seguridad** (círculo crítico de falla).

In [ ]:
# ── Grilla de búsqueda ────────────────────────────────────────────────────
xc_vals = np.linspace(15, 45, 25)
yc_vals = np.linspace(10, 30, 25)
R_vals  = np.linspace(10, 40, 25)

FS_min   = np.inf
xc_crit = yc_crit = R_crit = None
n_validos = 0

total = len(xc_vals) * len(yc_vals) * len(R_vals)
print(f'Buscando círculo crítico...')
print(f'  Grilla: {len(xc_vals)} × {len(yc_vals)} × {len(R_vals)} = {total:,} combinaciones')

for xc in xc_vals:
    for yc in yc_vals:
        for R in R_vals:
            FS, _, _ = factor_seguridad_bishop(xc, yc, R, n=n_dovelas)
            if FS is not None:
                n_validos += 1
                if FS < FS_min:
                    FS_min   = FS
                    xc_crit  = xc
                    yc_crit  = yc
                    R_crit   = R

print(f'  Círculos válidos encontrados: {n_validos:,}')
print()
print('=' * 52)
print('  CÍRCULO CRÍTICO DE FALLA')
print('=' * 52)
print(f'  xc = {xc_crit:.2f} m')
print(f'  yc = {yc_crit:.2f} m')
print(f'  R  = {R_crit:.2f} m')
print(f'  FS = {FS_min:.4f}')
print('=' * 52)

# Recalcular con el círculo crítico
FS_final, dovelas, hist_FS = factor_seguridad_bishop(
    xc_crit, yc_crit, R_crit, n=n_dovelas)

# Interpretación
if FS_final >= 1.5:
    estado = 'ESTABLE (FS ≥ 1.5)'
elif FS_final >= 1.2:
    estado = 'MARGINALMENTE ESTABLE (1.2 ≤ FS < 1.5)'
else:
    estado = 'INESTABLE (FS < 1.2)'

print(f'\n  Estado del talud: {estado}')

## Tabla de resultados por dovela

In [ ]:
header = (f"{'Dov':>4} {'xm [m]':>8} {'h [m]':>7} {'W [kN/m]':>10} "
          f"{'α [°]':>7} {'sin α':>7} {'cos α':>7} "
          f"{'σn [kPa]':>10} {'τf [kPa]':>10}")
print(header)
print('-' * len(header))

for i, d in enumerate(dovelas):
    print(f"{i+1:>4} {d['x']:>8.2f} {d['h']:>7.2f} {d['W']:>10.2f} "
          f"{d['alpha']:>7.2f} {d['sin_a']:>7.4f} {d['cos_a']:>7.4f} "
          f"{d['sigma']:>10.2f} {d['tau']:>10.2f}")

print('-' * len(header))
print(f'\n  FS = {FS_final:.4f}   |   Convergencia: {len(hist_FS)-1} iteraciones')

## Gráficas

### Gráfica 1 — Envolvente de Mohr-Coulomb con puntos de esfuerzo por dovela

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# Envolvente
sigma_max_d = max(d['sigma'] for d in dovelas)
sv = np.linspace(0, sigma_max_d * 1.3, 300)
tv = c_prima + sv * np.tan(phi)
ax.plot(sv, tv, 'k-', lw=2.5,
        label=rf"$\tau_f = c' + \sigma'_n \tan\phi'$   (c'={c_prima} kPa, φ'={phi_deg}°)")
ax.fill_between(sv, 0, tv, alpha=0.07, color='green', label='Zona segura (resistencia)')

# Puntos de cada dovela coloreados por ángulo α
sp = np.array([d['sigma'] for d in dovelas])
tp = np.array([d['tau']   for d in dovelas])
ap = np.array([d['alpha'] for d in dovelas])
sc = ax.scatter(sp, tp, c=ap, cmap='RdYlGn_r', s=70,
                zorder=5, edgecolors='k', linewidths=0.6,
                label='Dovelas  (color → α [°])')
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Ángulo α [°]')

# Annotate intercept
ax.axhline(c_prima, color='gray', linestyle=':', lw=1.2)
ax.text(sv[-1]*0.02, c_prima + 1, f"c' = {c_prima} kPa", color='gray', fontsize=9)
ax.axhline(0, color='k', lw=0.5)
ax.axvline(0, color='k', lw=0.5)

ax.set_xlabel(r"Esfuerzo normal efectivo $\sigma'_n$ [kPa]")
ax.set_ylabel(r'Resistencia al corte $\tau_f$ [kPa]')
ax.set_title('Gráfica 1 — Envolvente de Mohr-Coulomb con puntos de esfuerzo por dovela')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### Gráfica 2 — Geometría del talud con círculo crítico y dovelas

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

# ── Perfil del terreno ────────────────────────────────────────────────────
x_perf = np.linspace(0, x_max, 600)
y_perf = perfil_terreno(x_perf)
ax.fill_between(x_perf, 0, y_perf, color='#c8a96e', alpha=0.55, label='Suelo')
ax.plot(x_perf, y_perf, 'k-', lw=2, label='Perfil del terreno')
ax.axhline(0, color='saddlebrown', lw=1, linestyle='--', alpha=0.4)

# ── Círculo crítico ───────────────────────────────────────────────────────
theta = np.linspace(0, 2 * np.pi, 600)
ax.plot(xc_crit + R_crit * np.cos(theta),
        yc_crit + R_crit * np.sin(theta),
        'r-', lw=2, label=f'Círculo crítico  FS = {FS_final:.3f}')
ax.plot(xc_crit, yc_crit, 'r+', ms=14, mew=2)

# ── Dovelas ────────────────────────────────────────────────────────────────
xl_c, xr_c = interseccion_circulo_base(xc_crit, yc_crit, R_crit)
xl_c = max(xl_c, 0);  xr_c = min(xr_c, x_max)
dx_d = (xr_c - xl_c) / n_dovelas

for i, d in enumerate(dovelas):
    rect = patches.Rectangle(
        (d['x'] - dx_d/2, d['y_fal']), dx_d, d['h'],
        linewidth=0.7, edgecolor='navy', facecolor='steelblue', alpha=0.30
    )
    ax.add_patch(rect)
    if i % 4 == 0:
        ax.text(d['x'], d['y_ter'] + 0.25, str(i+1),
                ha='center', va='bottom', fontsize=7, color='navy', fontweight='bold')

# ── Radio anotado ─────────────────────────────────────────────────────────
ax.annotate('', xy=(xc_crit, yc_crit - R_crit),
            xytext=(xc_crit, yc_crit),
            arrowprops=dict(arrowstyle='->', color='red', lw=1.8))
ax.text(xc_crit + 0.5, yc_crit - R_crit/2,
        f'R = {R_crit:.1f} m', color='red', fontsize=9)

# ── Cota de altura ────────────────────────────────────────────────────────
ax.annotate('', xy=(0, H), xytext=(0, 0),
            arrowprops=dict(arrowstyle='<->', color='k'))
ax.text(-1.8, H/2, f'H = {H:.0f} m',
        ha='center', va='center', rotation=90, fontsize=9)

ax.set_xlim(-3, x_max + 3)
ax.set_ylim(-3, yc_crit + R_crit + 3)
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.set_title(f'Gráfica 2 — Geometría del talud y círculo crítico de falla\n'
             f'Centro: ({xc_crit:.1f}, {yc_crit:.1f}) m   R = {R_crit:.1f} m   '
             f'FS = {FS_final:.4f}')
ax.legend(loc='upper right', fontsize=9)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

### Gráfica 3 — Distribución de esfuerzos a lo largo de la superficie de falla

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x_d     = [d['x']    for d in dovelas]
sigma_d = [d['sigma'] for d in dovelas]
tau_d   = [d['tau']   for d in dovelas]

# ── Esfuerzo normal efectivo ──────────────────────────────────────────────
ax1 = axes[0]
ax1.fill_between(x_d, sigma_d, alpha=0.25, color='royalblue')
ax1.plot(x_d, sigma_d, 'o-', color='royalblue', lw=2, ms=5,
         label=r"$\sigma'_n$ [kPa]")
ax1.set_xlabel('Posición x [m]')
ax1.set_ylabel(r"$\sigma'_n$ [kPa]")
ax1.set_title('Esfuerzo normal efectivo - base de cada dovela')
ax1.legend()

# ── Resistencia al corte ──────────────────────────────────────────────────
ax2 = axes[1]
ax2.fill_between(x_d, tau_d, alpha=0.25, color='tomato')
ax2.plot(x_d, tau_d, 's-', color='tomato', lw=2, ms=5,
         label=r"$\tau_f = c' + \sigma'_n \tan\phi'$")
ax2.axhline(c_prima, color='gray', linestyle=':', lw=1.5,
            label=f"Cohesión pura  c' = {c_prima} kPa")
ax2.set_xlabel('Posición x [m]')
ax2.set_ylabel(r'$\tau_f$ [kPa]')
ax2.set_title('Resistencia al corte - base de cada dovela')
ax2.legend()

fig.suptitle('Gráfica 3 — Distribución de esfuerzos a lo largo de la superficie de falla',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

### Gráfica 4 — Mapa de calor del FS (grilla de centros)

In [ ]:
# Mapa de calor: xc vs yc con R = R_crit fijo
nx_m, ny_m = 30, 30
xc_m = np.linspace(15, 50, nx_m)
yc_m = np.linspace(10, 30, ny_m)
FS_m = np.full((ny_m, nx_m), np.nan)

for j, yc in enumerate(yc_m):
    for i, xc in enumerate(xc_m):
        fs, _, _ = factor_seguridad_bishop(xc, yc, R_crit, n=15)
        if fs is not None:
            FS_m[j, i] = fs

fig, ax = plt.subplots(figsize=(10, 6))

FS_p = np.clip(FS_m, 0.8, 4.0)
im = ax.contourf(xc_m, yc_m, FS_p, levels=20, cmap='RdYlGn')
cs = ax.contour(xc_m, yc_m, FS_p,
                levels=[1.0, 1.2, 1.5, 2.0, 2.5, 3.0],
                colors='k', linewidths=0.9)
ax.clabel(cs, fmt='FS=%.1f', fontsize=8)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Factor de Seguridad FS')

ax.plot(xc_crit, yc_crit, 'r*', ms=16,
        label=f'Círculo crítico ({xc_crit:.1f}, {yc_crit:.1f})\nFS = {FS_min:.4f}')
ax.legend(fontsize=9)

ax.set_xlabel('Centro $x_c$ [m]')
ax.set_ylabel('Centro $y_c$ [m]')
ax.set_title(f'Gráfica 4 — Mapa de calor del FS (R = {R_crit:.1f} m fijo)')
plt.tight_layout()
plt.show()

### Gráfica 5 — Convergencia iterativa de Bishop Simplificado

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

iters = np.arange(len(hist_FS))

# ── FS por iteración ──────────────────────────────────────────────────────
ax1 = axes[0]
ax1.plot(iters, hist_FS, 'o-', color='steelblue', lw=2, ms=8)
ax1.axhline(FS_final, color='red', linestyle='--', lw=1.5,
            label=f'FS convergido = {FS_final:.4f}')
ax1.set_xlabel('Iteración')
ax1.set_ylabel('Factor de Seguridad FS')
ax1.set_title('Convergencia del FS')
ax1.set_xticks(iters)
ax1.legend()

# ── Error relativo por iteración ──────────────────────────────────────────
ax2 = axes[1]
errores = [abs(hist_FS[k+1] - hist_FS[k]) for k in range(len(hist_FS)-1)]
ax2.semilogy(np.arange(1, len(hist_FS)), errores,
             's-', color='tomato', lw=2, ms=8)
ax2.axhline(1e-6, color='gray', linestyle=':', lw=1.5,
            label='Tolerancia = 1×10⁻⁶')
ax2.set_xlabel('Iteración')
ax2.set_ylabel('|ΔFS|  (escala log)')
ax2.set_title('Error relativo entre iteraciones')
ax2.set_xticks(np.arange(1, len(hist_FS)))
ax2.legend()

fig.suptitle('Gráfica 5 — Proceso iterativo de convergencia  (Bishop Simplificado)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## Resumen de resultados e interpretación

In [ ]:
sigma_max_r = max(d['sigma'] for d in dovelas)
tau_max_r   = max(d['tau']   for d in dovelas)
W_total     = sum(d['W']     for d in dovelas)
n_iter      = len(hist_FS) - 1

print('=' * 60)
print('  RESUMEN COMPLETO DE RESULTADOS')
print('=' * 60)
print()
print('  PROPIEDADES DEL SUELO')
print(f"    γ = {gamma} kN/m³  |  c' = {c_prima} kPa  |  φ' = {phi_deg}°")
print()
print('  CÍRCULO CRÍTICO DE FALLA')
print(f'    Centro: xc = {xc_crit:.2f} m,  yc = {yc_crit:.2f} m')
print(f'    Radio:  R  = {R_crit:.2f} m')
print()
print('  FACTOR DE SEGURIDAD  (Bishop Simplificado)')
print(f'    FS mínimo = {FS_final:.4f}')
print(f'    Converge en {n_iter} iteraciones')
print()
print('  ESFUERZOS EN LA SUPERFICIE DE FALLA')
print(f'    σ\'n máx = {sigma_max_r:.2f} kPa')
print(f'    τf  máx = {tau_max_r:.2f} kPa')
print(f'    Peso total dovelas = {W_total:.2f} kN/m')
print()
print('  INTERPRETACIÓN')
print(f'    FS = {FS_final:.4f}  →  {estado}')
if FS_final >= 1.5:
    print(f'    La resistencia al corte supera la fuerza actuante en un {(FS_final-1)*100:.1f}%.')
    print('    El talud cumple el criterio mínimo de seguridad (FS ≥ 1.5).')
elif FS_final >= 1.2:
    print('    El talud está marginalmente estable. Se recomienda revisión.')
else:
    print('    ATENCIÓN: El talud podría ser inestable. Se requiere intervención.')
print()
print('  CRITERIO DE MOHR-COULOMB APLICADO')
print(f"    τf = c' + σ'n·tan(φ') = {c_prima} + σ'n·tan({phi_deg}°)")
print(f"    τf = {c_prima} + σ'n × {np.tan(phi):.4f}")
print('=' * 60)